# Qwen2.5-VL-3B grounding test on Colab (ball position, motor angles, marker position)

Three tests on the Track 4 grounding bundle (`grounding_bundle.zip`, built by
`host_software/ml_jetson_vla/deployment/build_colab_grounding_bundle.py`):

- **BALL**: ask for the ball's pixel position on every bundled frame; score in the telemetry frame against `touch_x`/`touch_y`.
- **MOTOR**: ask for `theta_a`, `theta_b`, `theta_c` in degrees; score against the logged theta columns.
- **MARKER**: colour-marker prompt on colour-command frames only (the Jetson sweep's convention); score against `target_x`/`target_y`.

Model: `Qwen/Qwen2.5-VL-3B-Instruct` @ `66285546d2b821cf421d4f5eb2576359d3770cd3`, bf16, min/max pixels 50176/200704, max_new_tokens 24, greedy.

**Inputs you must upload to Drive first**: `MyDrive/VRI_2026/colab_bundle/grounding_bundle.zip` and `MyDrive/VRI_2026/colab_bundle/BUNDLE_MANIFEST.txt`.

**Runtime**: GPU (A100, L4 or T4). Note T4 has no native bf16; the run is kept at bf16 as specified, so expect it to be slow or fail on T4 (see the GPU cell).

**Cost**: the bundle has roughly 4 generate calls per colour frame and 2 per other frame; the run checkpoints after every call and resumes by skipping completed (frame, test) pairs.

## (a) GPU check and versions

In [ ]:
!pip install -q "transformers==5.17.0" qwen-vl-utils

In [ ]:
import importlib.metadata as md_meta
import time

T0 = time.time()
import torch
import transformers
import qwen_vl_utils  # noqa: F401  (import check; the version is printed below)

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("qwen-vl-utils", md_meta.version("qwen-vl-utils"))
assert transformers.__version__ == "5.17.0", f"expected transformers 5.17.0, got {transformers.__version__}"

In [ ]:
ACCEPTED_GPUS = ("A100", "L4", "T4")
assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > GPU."
gpu_name = torch.cuda.get_device_name(0)
print("device:", gpu_name)
if not any(tag in gpu_name for tag in ACCEPTED_GPUS):
    raise RuntimeError(f"GPU {gpu_name!r} is not one of {ACCEPTED_GPUS}")
if "T4" in gpu_name:
    print("WARNING: T4 has no native bf16; bf16 may be emulated and slow. Kept at bf16 as specified.")

## (b) Mount Drive, copy the bundle, verify sha256, unzip

In [ ]:
import hashlib
import json
import os
import shutil
import zipfile

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
DRIVE_BUNDLE_DIR = "/content/drive/MyDrive/VRI_2026/colab_bundle"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/VRI_2026/qwen_grounding_results"
LOCAL_DIR = "/content/grounding"
ZIP_NAME = "grounding_bundle.zip"
BUNDLE_MANIFEST_NAME = "BUNDLE_MANIFEST.txt"
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

local_zip = os.path.join(LOCAL_DIR, ZIP_NAME)
shutil.copy(os.path.join(DRIVE_BUNDLE_DIR, ZIP_NAME), local_zip)
shutil.copy(os.path.join(DRIVE_BUNDLE_DIR, BUNDLE_MANIFEST_NAME), os.path.join(LOCAL_DIR, BUNDLE_MANIFEST_NAME))


def sha256_of(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected_sha = None
with open(os.path.join(LOCAL_DIR, BUNDLE_MANIFEST_NAME), encoding="utf-8") as fh:
    for line in fh:
        if line.startswith("zip_sha256:"):
            expected_sha = line.split(":", 1)[1].strip()
assert expected_sha, "zip_sha256 missing from BUNDLE_MANIFEST.txt"
actual_sha = sha256_of(local_zip)
print("expected sha256:", expected_sha)
print("actual   sha256:", actual_sha)
assert actual_sha == expected_sha, "bundle sha256 mismatch: re-upload grounding_bundle.zip"

with zipfile.ZipFile(local_zip) as zf:
    zf.extractall(LOCAL_DIR)
BUNDLE = os.path.join(LOCAL_DIR, "bundle")
manifest = pd.read_csv(os.path.join(BUNDLE, "manifest.csv"))
print("manifest rows:", len(manifest))
print(manifest["audio_command"].value_counts().to_string())

## (c) Load the model at the pinned revision (bf16)

In [ ]:
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
REVISION = "66285546d2b821cf421d4f5eb2576359d3770cd3"
MIN_PIXELS = 50176
MAX_PIXELS = 200704
MAX_NEW_TOKENS = 24

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    low_cpu_mem_usage=True,
).eval()
processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=REVISION, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
)
loaded_commit = getattr(model.config, "_commit_hash", None)
print("pinned revision:", REVISION)
print("config _commit_hash:", loaded_commit if loaded_commit is not None else "not exposed by this transformers build")
if loaded_commit is not None and loaded_commit != REVISION:
    raise RuntimeError(f"loaded commit {loaded_commit} differs from pinned {REVISION}")
print("parameter dtype:", next(model.parameters()).dtype)

## (d) Prompt sets

The COLOR_COMMANDS, PROMPT_TEMPLATE (baseline variant) and build_prompt logic are copied from
`ml_jetson_vla/core/minimal_vlm_policy.py` (not paraphrased). The local check asserts equality with that module.
The MOTOR prompt is new; it names the joints, states degrees, and describes no geometry.

In [ ]:
# Copied from ml_jetson_vla/core/minimal_vlm_policy.py: COLOR_COMMANDS, PROMPT_TEMPLATE (baseline), build_prompt.
COLOR_COMMANDS = {
    "go_red": "red marker",
    "go_green": "green marker",
    "go_yellow": "yellow marker",
    "go_black": "black marker",
}
PROMPT_TEMPLATE = (
    "You are looking straight down at a small 3-DOF ball-balancing platform. "
    "Find the {target_label} in this image. "
    "Respond with ONLY a single JSON object and nothing else, in exactly this form: "
    '{{"target_point_xy": [x, y]}} '
    "where x and y are pixel coordinates in THIS image (x in [0, {width}], y in [0, {height}])."
)


def build_prompt(instruction, image_w: int, image_h: int):
    """Returns (prompt_text, target_label). Non-colour instructions (and None) get the 'ball' label,
    which is the ball-position prompt."""
    label = COLOR_COMMANDS[instruction] if instruction in COLOR_COMMANDS else "ball"
    return PROMPT_TEMPLATE.format(target_label=label, width=image_w, height=image_h), label


# New prompt (no sweep reference exists for motor angles).
MOTOR_PROMPT = (
    "Estimate the current joint angles of this 3-DOF ball-balancing platform: theta_a, theta_b and theta_c. "
    "Respond with ONLY a single JSON object and nothing else, in exactly this form: "
    '{"theta_a": <degrees>, "theta_b": <degrees>, "theta_c": <degrees>} '
    "where each value is the angle of that joint in degrees."
)
MOTOR_KEYS = ("theta_a", "theta_b", "theta_c")

In [ ]:
# Parsers. parse_point_output mirrors parse_minimal_baseline_output steps 2-3 (the asked JSON contract,
# then Qwen's point_2d / bbox_2d list). Its PaliGemma <loc> step is omitted: Qwen does not emit it.
import json
import re
from typing import List, Optional, Tuple

_JSON_OBJECT_RE = re.compile(r"\{.*?\}", re.DOTALL)
_QWEN_POINT_LIST_RE = re.compile(r"\[\s*\{.*?\}\s*\]", re.DOTALL)


def parse_point_output(raw_text: str) -> Tuple[bool, Optional[float], Optional[float], str]:
    stripped = raw_text.strip()
    match = _JSON_OBJECT_RE.search(stripped)
    if match:
        try:
            obj = json.loads(match.group(0))
            pt = obj.get("target_point_xy")
            if isinstance(pt, (list, tuple)) and len(pt) == 2:
                return True, float(pt[0]), float(pt[1]), "asked_json_contract"
        except (json.JSONDecodeError, TypeError, ValueError, AttributeError):
            pass
    match = _QWEN_POINT_LIST_RE.search(stripped)
    if match:
        try:
            objs = json.loads(match.group(0))
            if isinstance(objs, list) and objs:
                first = objs[0]
                if "point_2d" in first:
                    x, y = first["point_2d"]
                    return True, float(x), float(y), "qwen_point_2d"
                if "bbox_2d" in first:
                    x1, y1, x2, y2 = first["bbox_2d"]
                    return True, (float(x1) + float(x2)) / 2.0, (float(y1) + float(y2)) / 2.0, "qwen_bbox_2d_center"
        except (json.JSONDecodeError, TypeError, ValueError, KeyError, IndexError):
            pass
    return False, None, None, "no_known_convention_matched"


def parse_motor_output(raw_text: str) -> Tuple[bool, Optional[List[float]], str]:
    match = _JSON_OBJECT_RE.search(raw_text.strip())
    if not match:
        return False, None, "no_json_object"
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError:
        return False, None, "json_decode_error"
    try:
        values = [float(obj[k]) for k in MOTOR_KEYS]
    except (KeyError, TypeError, ValueError):
        return False, None, "missing_or_nonnumeric_key"
    return True, values, "theta_json"

## (e) Run, one generate call at a time, checkpointing to Drive after every call

Each (frame, test) pair is one JSONL line appended to Drive and fsynced before the next call. Re-running skips pairs already in the file.
`MAX_FRAMES` limits the frame rows for a smoke run; leave it `None` for the full run.

In [ ]:
from PIL import Image

MAX_FRAMES = None  # e.g. 20 for a smoke run; None for the full run
CHECKPOINT_PATH = os.path.join(DRIVE_RESULTS_DIR, "checkpoint.jsonl")
COLOR_SET = set(COLOR_COMMANDS)


def load_done_keys(path: str) -> set:
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                if line.strip():
                    rec = json.loads(line)
                    done.add((rec["image_file"], rec["test"]))
    return done


def generate(image_path: str, prompt: str) -> Tuple[str, float, List[int]]:
    # Message shape copied from deployment/qwen_vl_smoke_test.py generate_qwen_vl(), including its
    # per-image max_pixels entry, so the vision tower sees the same input the Jetson sweep did.
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path, "max_pixels": 1280 * 28 * 28},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt",
    ).to("cuda")
    grid = inputs["image_grid_thw"]
    patch = int(getattr(processor.image_processor, "patch_size", 14))
    model_input_hw = [int(grid[0][1]) * patch, int(grid[0][2]) * patch]
    start = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    latency = time.time() - start
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    output_text = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False,
    )[0]
    return output_text, latency, model_input_hw


def append_record(rec: dict) -> None:
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(rec) + "\n")
        fh.flush()
        os.fsync(fh.fileno())


work_manifest = manifest.head(MAX_FRAMES) if MAX_FRAMES is not None else manifest
done = load_done_keys(CHECKPOINT_PATH)
print("already checkpointed:", len(done), "| frames this run:", len(work_manifest))

written = 0
skipped = 0
run_start = time.time()
for row in work_manifest.itertuples(index=False):
    image_path = os.path.join(BUNDLE, row.image_file)
    img_w, img_h = Image.open(image_path).size
    tests = ["ball", "motor"] + (["marker"] if row.audio_command in COLOR_SET else [])
    for test in tests:
        key = (row.image_file, test)
        if key in done:
            skipped += 1
            continue
        if test == "ball":
            prompt, _ = build_prompt(None, img_w, img_h)
        elif test == "marker":
            prompt, _ = build_prompt(row.audio_command, img_w, img_h)
        else:
            prompt = MOTOR_PROMPT
        raw_text, latency, model_input_hw = generate(image_path, prompt)
        if test == "motor":
            ok, values, reason = parse_motor_output(raw_text)
            x_px = y_px = None
            theta = values
        else:
            ok, x_px, y_px, reason = parse_point_output(raw_text)
            theta = None
        append_record({
            "image_file": row.image_file,
            "session": row.session,
            "frame_index": int(row.frame_index),
            "audio_command": row.audio_command,
            "test": test,
            "raw_text": raw_text,
            "parse_ok": ok,
            "parse_reason": reason,
            "x_px": x_px,
            "y_px": y_px,
            "theta": theta,
            "image_w": img_w,
            "image_h": img_h,
            "model_input_hw": model_input_hw,
            "latency_s": latency,
            "revision": REVISION,
        })
        done.add(key)
        written += 1
        if written % 100 == 0:
            print(f"written {written} | skipped {skipped} | elapsed {time.time() - run_start:.0f}s")
print("done this run: written", written, "| skipped", skipped)

## (f) Scoring helpers and aggregation

Each helper mirrors a function in `deployment/score_minimal_baseline_offline.py` (or `core/minimal_vlm_policy.py`), named in its comment.
The local check compares these against the scorer's own functions on synthetic points.

In [ ]:
import cv2
import numpy as np

PLATFORM_W_MM = 187.5
PLATFORM_H_MM = 142.0
TOLERANCE_MM = 20.0


def raw_px_to_mm(homography_mm_to_px, x_px: float, y_px: float) -> Tuple[float, float]:
    # Mirrors score_minimal_baseline_offline.raw_px_to_mm: inverts the mm->px homography, then maps the point.
    m_inv = np.linalg.inv(np.asarray(homography_mm_to_px, dtype=np.float64))
    pt = np.array([[[x_px, y_px]]], dtype=np.float32)
    mm_pt = cv2.perspectiveTransform(pt, m_inv.astype(np.float32))
    return float(mm_pt[0][0][0]), float(mm_pt[0][0][1])


def touch_frame_to_manifest_mm(tx: float, ty: float, w: float = PLATFORM_W_MM, h: float = PLATFORM_H_MM) -> Tuple[float, float]:
    # Mirrors score_minimal_baseline_offline.touch_frame_to_manifest_mm (telemetry -> manifest frame).
    return w / 2.0 - float(tx), h / 2.0 + float(ty)


def manifest_mm_to_telemetry(mx: float, my: float, w: float = PLATFORM_W_MM, h: float = PLATFORM_H_MM) -> Tuple[float, float]:
    # Inverse of touch_frame_to_manifest_mm: the same W/2 - x, H/2 + y relation applied the other way.
    return w / 2.0 - mx, my - h / 2.0


def to_raw_px_model_input(x: float, y: float, model_input_hw, raw_hw) -> Tuple[float, float]:
    # Mirrors core/minimal_vlm_policy.to_raw_px for coord_space "model_input".
    raw_h, raw_w = raw_hw
    mi_h, mi_w = model_input_hw
    return x * raw_w / float(mi_w), y * raw_h / float(mi_h)


def score_point_in_telemetry_frame(homography_mm_to_px, x: float, y: float, model_input_hw, raw_hw, truth_tx: float, truth_ty: float) -> dict:
    # Mirrors the primary path of score_minimal_baseline_offline.score_prediction (error_mm, hit).
    rx, ry = to_raw_px_model_input(x, y, model_input_hw, raw_hw)
    mx, my = raw_px_to_mm(homography_mm_to_px, rx, ry)
    pred_tx, pred_ty = manifest_mm_to_telemetry(mx, my)
    err = float(np.hypot(pred_tx - truth_tx, pred_ty - truth_ty))
    return {"pred_tx_mm": pred_tx, "pred_ty_mm": pred_ty, "error_mm": err, "hit": bool(err <= TOLERANCE_MM)}

In [ ]:
COMMAND_GROUPS = ["colour", "hold", "forward", "left", "right", "backward", "stop"]


def command_group(cmd: str) -> str:
    return "colour" if cmd in COLOR_SET else cmd


def homography_of(row) -> Optional[list]:
    value = row.homography_mm_to_px
    return json.loads(value) if isinstance(value, str) and value.strip() else None


def summarize_points(entries: list) -> dict:
    n = len(entries)
    parsed = [e for e in entries if e["parse_ok"]]
    scored = [e for e in parsed if e["error_mm"] is not None]
    errs = [e["error_mm"] for e in scored]
    hits = [e["hit"] for e in scored]
    return {
        "n": n,
        "parse_rate": (len(parsed) / n) if n else None,
        "n_scored": len(scored),
        "no_homography": sum(1 for e in parsed if e["error_mm"] is None),
        "mean_error_mm": float(np.mean(errs)) if errs else None,
        "median_error_mm": float(np.median(errs)) if errs else None,
        "hit_rate_20mm": (sum(hits) / len(hits)) if hits else None,
        "hits": int(sum(hits)),
    }


def summarize_motor(entries: list) -> dict:
    n = len(entries)
    parsed = [e for e in entries if e["parse_ok"]]
    out = {"n": n, "parse_rate": (len(parsed) / n) if n else None, "n_scored": len(parsed)}
    if parsed:
        err = np.abs(np.array([e["abs_err"] for e in parsed], dtype=float))
        for j, key in enumerate(MOTOR_KEYS):
            out[f"mae_{key}_deg"] = float(err[:, j].mean())
            out[f"within_2deg_{key}"] = float((err[:, j] <= 2.0).mean())
        out["within_2deg_all_joints"] = float((err <= 2.0).all(axis=1).mean())
    else:
        for key in MOTOR_KEYS:
            out[f"mae_{key}_deg"] = None
            out[f"within_2deg_{key}"] = None
        out["within_2deg_all_joints"] = None
    return out


records = [json.loads(line) for line in open(CHECKPOINT_PATH, encoding="utf-8") if line.strip()]
manifest_by_file = {r.image_file: r for r in manifest.itertuples(index=False)}

ball_entries, marker_entries, motor_entries = [], [], []
for rec in records:
    row = manifest_by_file[rec["image_file"]]
    cmd = rec["audio_command"]
    group = command_group(cmd)
    if rec["test"] == "motor":
        entry = {"command": cmd, "group": group, "parse_ok": rec["parse_ok"]}
        if rec["parse_ok"]:
            truth = [float(row.theta_a), float(row.theta_b), float(row.theta_c)]
            entry["abs_err"] = [abs(p - t) for p, t in zip(rec["theta"], truth)]
        motor_entries.append(entry)
        continue
    H = homography_of(row)
    entry = {
        "command": cmd, "group": group, "parse_ok": rec["parse_ok"],
        "touch_suspect": bool(row.touch_suspect), "settled": not bool(row.target_transition),
        "error_mm": None, "hit": None,
    }
    if rec["parse_ok"] and H is not None:
        raw_hw = (rec["image_h"], rec["image_w"])
        if rec["test"] == "ball":
            truth_tx, truth_ty = float(row.touch_x), float(row.touch_y)
        else:
            truth_tx, truth_ty = float(row.target_x), float(row.target_y)
        s = score_point_in_telemetry_frame(H, rec["x_px"], rec["y_px"], rec["model_input_hw"], raw_hw, truth_tx, truth_ty)
        entry["error_mm"], entry["hit"] = s["error_mm"], s["hit"]
    if rec["test"] == "ball":
        ball_entries.append(entry)
    else:
        marker_entries.append(entry)

ball_all = {g: summarize_points([e for e in ball_entries if e["group"] == g]) for g in COMMAND_GROUPS}
ball_all["ALL"] = summarize_points(ball_entries)
ball_unsuspect = summarize_points([e for e in ball_entries if not e["touch_suspect"]])
motor_res = {g: summarize_motor([e for e in motor_entries if e["group"] == g]) for g in COMMAND_GROUPS}
motor_res["ALL"] = summarize_motor(motor_entries)
marker_res = {c: summarize_points([e for e in marker_entries if e["command"] == c]) for c in COLOR_COMMANDS}
marker_res["ALL"] = summarize_points(marker_entries)
marker_settled = summarize_points([e for e in marker_entries if e["settled"]])
print("scored records:", len(records))

## (g) Results to Drive and plain verdicts

In [ ]:
results = {
    "model": MODEL_ID,
    "revision": REVISION,
    "loaded_commit": loaded_commit,
    "dtype": "bf16",
    "max_new_tokens": MAX_NEW_TOKENS,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "tolerance_mm": TOLERANCE_MM,
    "bundle_zip_sha256": actual_sha,
    "frames_in_bundle": int(len(manifest)),
    "ball": {"by_group": ball_all, "excluding_touch_suspect": ball_unsuspect},
    "motor": {"by_group": motor_res},
    "marker": {"by_command": marker_res, "settled_only": marker_settled},
}
with open(os.path.join(DRIVE_RESULTS_DIR, "results.json"), "w", encoding="utf-8") as fh:
    json.dump(results, fh, indent=2)

summary_rows = []
for group, s in ball_all.items():
    summary_rows.append({"test": "ball", "group": group, **s})
for group, s in motor_res.items():
    summary_rows.append({"test": "motor", "group": group, **s})
for cmd, s in marker_res.items():
    summary_rows.append({"test": "marker", "group": cmd, **s})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(DRIVE_RESULTS_DIR, "summary_table.csv"), index=False)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
print(summary_df.to_string(index=False))

b = ball_all["ALL"]
print(f"\nVERDICT BALL: hit@20mm {b['hits']}/{b['n_scored']} ({b['hit_rate_20mm']}), "
      f"mean error {b['mean_error_mm']} mm, parse rate {b['parse_rate']}, n={b['n']}")
m = motor_res["ALL"]
print(f"VERDICT MOTOR: MAE theta_a/b/c = {m['mae_theta_a_deg']}/{m['mae_theta_b_deg']}/{m['mae_theta_c_deg']} deg, "
      f"all-joints within 2 deg = {m['within_2deg_all_joints']}, parse rate {m['parse_rate']}, n={m['n']}")
k = marker_res["ALL"]
print(f"VERDICT MARKER: hit@20mm {k['hits']}/{k['n_scored']} ({k['hit_rate_20mm']}), "
      f"mean error {k['mean_error_mm']} mm, parse rate {k['parse_rate']}, n={k['n']}")

## (h) Checkpoint count and wall time

In [ ]:
n_checkpoint_lines = sum(1 for line in open(CHECKPOINT_PATH, encoding="utf-8") if line.strip())
print("checkpoint lines:", n_checkpoint_lines)
print("wall time this runtime (s):", round(time.time() - T0, 1))